# Optimization basics

## Session 3: Random and multistart search

Use the Week 2 page in ELO (Session 3 section) and its exercise sheet.
Consult the [notation glossary](NotationGlossary.pdf) as needed;
it is a reference, not an additional assignment.
Sections A–D are the Session 3 route. **Stop before Later sessions: random search
and annealing.** No new dependency is required beyond NumPy and Matplotlib.

**Weekly timing:** Sessions 3 and 4 take place on the same day, with a
15-minute rest break. No homework or setup is required during that gap.
Complete this follow-up after both sessions and before Session 5 the next week.

In class, predict and inspect the supplied implementations. The follow-up uses
30 minutes of reading and 45 minutes of practice: 10 worksheet, 25 notebook,
10 discussion and your own explanation. O1 is included once; O2–O5 belong to later sessions.

Our questions: How many independent trials give a stated chance of entering a
useful region? How does local improvement change the use of an evaluation budget?
How does the answer depend on the objective?

In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print('Python', sys.version.split()[0])
print('NumPy', np.__version__)
print('Matplotlib', matplotlib.__version__)

## A. A success region and independent trials

All random variables below are defined on a probability space
$(\Omega,\mathcal F,\Pr)$, where $\Omega$ is the sample space, $\mathcal F$
is its sigma-algebra of events, and $\Pr$ is the probability measure.
Let $n\in\mathbb N=\{1,2,\ldots\}$ and $p\in[0,1]$. For iid random variables
$V_1,\ldots,V_n\sim\mathrm{Uniform}[0,1]$, define $E_i=\{V_i<p\}$.
These events are mutually independent and $\Pr(E_i)=p$. Hence
$$\Pr\!\left(\bigcup_{i=1}^n E_i\right)=1-(1-p)^n.$$
Derive this using the complementary event before running the cell (O1, S3.1).
For $p=0.1$, predict how many trials give probability at least $q=0.95$.
For $0<p,q<1$, the smallest trial count is
$$n_{\min}=\left\lceil\frac{\log(1-q)}{\log(1-p)}\right\rceil.$$
Here $\log$ is the natural logarithm and $\lceil u\rceil$ is the smallest
integer at least $u$. Zero trials mean no hit. The helper handles $p,q=0,1$
separately and returns `np.inf` if no finite count reaches the target.
It evaluates the interior formula in floating-point arithmetic; near an integer
boundary, rounding can affect the ceiling. Check the probabilities at the
returned count and the preceding count, as done below.

The simulation below approximates this model using pseudorandom numbers:
500 whole experiments, each with 60 draws.
For repetition $r\in\{1,\ldots,500\}$ and trial $i\in\{1,\ldots,60\}$,
write $V_i^{(r)}$ for the uniform draw; the model assumes all these variables
are independent. The array has shape `(500, 60)`: rows are whole experiments
and columns are trials. Python entry `[r-1, i-1]` represents one observed draw.
`hit_draws < p` marks individual hits. `maximum.accumulate(..., axis=1)`
records whether a row has had at least one hit by each trial count.
Finally, `mean(axis=0)` averages those Boolean results across the 500 rows.
At $n=29$, for example, the plotted fraction counts rows with a hit among
their first 29 entries. It estimates $1-(1-p)^{29}$, not the single-draw
probability $p$. Budgets within a row use prefixes of that row, so points
on the empirical curve are dependent.

In [ ]:
def success_probability(p, n):
    n = np.asarray(n)
    if not 0 <= p <= 1 or np.any(n < 0) or not np.issubdtype(n.dtype, np.integer):
        raise ValueError('use a probability p and nonnegative integer trial counts')
    if p == 1:
        return (n > 0).astype(float)
    return -np.expm1(n * np.log1p(-p))

def required_trials(p, q):
    if not (0 <= p <= 1 and 0 <= q <= 1):
        raise ValueError('p and q must be probabilities')
    if q == 0:
        return 0
    if p == 1:
        return 1
    if p == 0 or q == 1:
        return np.inf
    return int(np.ceil(np.log1p(-q) / np.log1p(-p)))

hit_probability = 0.1
desired_probability = 0.95
trial_counts = np.arange(1, 61)
hit_draws = np.random.default_rng(3003).uniform(size=(500, 60))
observed_success = np.maximum.accumulate(hit_draws < hit_probability, axis=1).mean(axis=0)
predicted_success = success_probability(hit_probability, trial_counts)
needed = required_trials(hit_probability, desired_probability)
print('Required trials:', needed)
print('Probability at n-1 and n:', success_probability(hit_probability, np.array([needed-1, needed])))
fig_success, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(trial_counts, predicted_success, label='Exact probability', color='#246a91')
ax.step(trial_counts, observed_success, where='post', label='500 repeated experiments', color='#b65e2f')
ax.axhline(desired_probability, color='black', linestyle='--', label='Target 0.95')
ax.set(xlabel='Independent trials n', ylabel='Probability of at least one hit', ylim=(0, 1.03))
ax.legend()
fig_success.tight_layout()
plt.show()

The model probability differs from a finite observed success fraction.
For $d\in\mathbb N$, let $D=\prod_{j=1}^d[a_j,b_j]$, with real $a_j<b_j$,
and let $A\subseteq D$ be a Borel success region. For $X\sim\mathrm{Uniform}(D)$,
$$p=\Pr(X\in A)=\frac{\lambda_d(A)}{\prod_{j=1}^d(b_j-a_j)},$$
where $\lambda_d$ is $d$-dimensional volume. This probability is usually
unknown for an objective threshold.

**Dimension prediction (S3.2):** Suppose an axis-aligned useful box
$A\subseteq D$ occupies a fraction $r=0.1$ of each coordinate interval.
Then $p=r^d$.
This useful box is a teaching example; it is not the exact sublevel set of the
Sphere or Rastrigin experiment below.

In [ ]:
dimensions = np.array([1, 2, 5])
volume_probabilities = 0.1**dimensions
for dimension, p in zip(dimensions, volume_probabilities):
    print(f'd={dimension}: useful-volume fraction={p:g}, trials for 95%={required_trials(p, 0.95)}')

## B. Objectives and a deterministic local search

Let $D=[-4,4]^2$, $x^*=(0.4,-0.6)$ and $z=x-x^*$ for $x\in D$.
Define continuous objectives $f_{\mathrm{Sphere}},f_{\mathrm{Rastrigin}}:D\to\mathbb R$ by
$$f_{\mathrm{Sphere}}(x)=\sum_{j=1}^2z_j^2,\qquad
f_{\mathrm{Rastrigin}}(x)=\sum_{j=1}^2[z_j^2+10(1-\cos(2\pi z_j))].$$
Here $z_j$ denotes coordinate $j$ and $\pi$ is the circle constant.
Every summand is nonnegative. Value zero forces $z=0$, so both have
optimal value $f_*:=\min_{x\in D}f(x)=0$ and unique minimizer $x^*$.
The Python variable `session3_target` represents $x^*$.
The search routines receive the objective and domain, but not the target.
The target is used only inside the supplied objective and for validation/plotting.

For the Sphere objective and threshold $\varepsilon=0.001$, success means
$f_{\mathrm{Sphere}}(x)\leq\varepsilon$. The success region is the disk
$\{x:\|x-x^*\|_2\leq\sqrt\varepsilon\}$, where
$\|v\|_2=\sqrt{v_1^2+v_2^2}$ for $v=(v_1,v_2)\in\mathbb R^2$.
This disk lies inside $D$ and has area $\pi\varepsilon$, whereas $D$ has
area 64. Thus one uniform candidate succeeds with probability
$p=\pi\varepsilon/64\approx0.00004909$. A complete 400-call uniform-search
run succeeds with probability $1-(1-p)^{400}\approx0.01944$.
This connects the earlier volume calculation to the actual benchmark.
A success count from 24 runs need not equal 24 times this probability.

In [ ]:
session3_target = np.array([0.4, -0.6])
session3_domain = np.array([[-4.0, 4.0], [-4.0, 4.0]])

def sphere(x):
    z = np.asarray(x) - session3_target
    return float(np.sum(z*z))

def rastrigin(x):
    z = np.asarray(x) - session3_target
    return float(np.sum(z*z + 10*(1 - np.cos(2*np.pi*z))))

grid = np.linspace(-4, 4, 241)
xx, yy = np.meshgrid(grid, grid)
zz_sphere = (xx-session3_target[0])**2 + (yy-session3_target[1])**2
zz_rastrigin = zz_sphere + 20 - 10*np.cos(2*np.pi*(xx-session3_target[0])) - 10*np.cos(2*np.pi*(yy-session3_target[1]))
fig_landscapes, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, zz, label in zip(axes, [zz_sphere, zz_rastrigin], ['Shifted Sphere', 'Shifted Rastrigin']):
    contour = ax.contourf(xx, yy, zz, levels=18, cmap='viridis')
    ax.plot(*session3_target, 'rx', label='Reference minimizer')
    ax.set(xlabel='x1', ylabel='x2', title=label, aspect='equal')
    ax.legend(fontsize=8)
    fig_landscapes.colorbar(contour, ax=ax, label='Objective')
fig_landscapes.tight_layout()
plt.show()

**Supplied local rule:** evaluate a start in the box, then poll coordinates in order
0,1,… in Python (coordinates 1,2,… in the mathematics), with directions − then +.
A poll tries these directions until an improvement is accepted or all directions
have been considered. Accept the first strict improvement and begin
a new poll at the same step. After a full poll without improvement, halve the
step. Reject infeasible proposals without evaluating them. Stop when the step
falls below the declared minimum or the local evaluation cap is exhausted.

The rule is deterministic given its start and settings. It is an introductory
coordinate search, not a certified local-minimum solver. The evaluation cap may
interrupt a poll. Infeasible proposals spend no objective calls; feasible
proposals spend one call even when rejected or previously visited. A full
unsuccessful poll supports only a statement about the tested directions at
that step size. It does not establish a local minimum in a neighbourhood.
Trace the one-dimensional example on the worksheet first.

In [ ]:
def _check_search_inputs(domain, budget):
    domain = np.asarray(domain, dtype=float)
    if domain.ndim != 2 or domain.shape[1] != 2 or domain.shape[0] == 0:
        raise ValueError('domain must have one [lower, upper] row per coordinate')
    if not np.all(np.isfinite(domain)) or np.any(domain[:, 0] >= domain[:, 1]):
        raise ValueError('domain bounds must be finite and increasing')
    if isinstance(budget, (bool, np.bool_)) or not isinstance(budget, (int, np.integer)) or budget < 1:
        raise ValueError('budget must be a positive integer')
    return domain

def uniform_search(f, domain, budget, rng):
    domain = _check_search_inputs(domain, budget)
    best_x, best_value = None, np.inf
    history = []
    for _ in range(budget):
        x = rng.uniform(domain[:, 0], domain[:, 1])
        value = float(f(x))  # One counted objective call per trial.
        if not np.isfinite(value):
            raise ValueError('the teaching objectives must be finite')
        if value < best_value:
            best_x, best_value = x.copy(), value
        history.append(best_value)
    return dict(x=best_x, value=best_value, history=np.array(history),
                evaluations=budget, starts=budget)

def coordinate_search(f, domain, start, budget, initial_step=0.5, min_step=0.001):
    domain = _check_search_inputs(domain, budget)
    x = np.asarray(start, dtype=float).copy()
    if x.shape != (domain.shape[0],) or not np.all(np.isfinite(x)):
        raise ValueError('start must contain one finite value per coordinate')
    if np.any(x < domain[:, 0]) or np.any(x > domain[:, 1]):
        raise ValueError('start must be feasible')
    if not (np.isfinite(initial_step) and np.isfinite(min_step) and 0 < min_step <= initial_step):
        raise ValueError('use finite steps with 0 < min_step <= initial_step')
    value = float(f(x))  # The starting point costs one evaluation.
    if not np.isfinite(value):
        raise ValueError('the teaching objectives must be finite')
    history = [value]
    proposals = 0
    step = initial_step
    while len(history) < budget and step >= min_step:
        improved = False
        for j in range(x.size):
            for direction in (-1, 1):
                if len(history) == budget:
                    break
                proposal = x.copy()
                proposal[j] += direction * step
                proposals += 1
                if np.any(proposal < domain[:, 0]) or np.any(proposal > domain[:, 1]):
                    continue
                proposed_value = float(f(proposal))
                if not np.isfinite(proposed_value):
                    raise ValueError('the teaching objectives must be finite')
                if proposed_value < value:
                    x, value = proposal, proposed_value
                    improved = True
                history.append(value)  # Include every evaluated rejected point.
                if improved:
                    break
            if improved or len(history) == budget:
                break
        if not improved:
            step *= 0.5
    reason = 'evaluation cap' if len(history) == budget else 'step threshold'
    return dict(x=x, value=value, history=np.array(history),
                evaluations=len(history), proposals=proposals, stop_reason=reason)

def multistart_search(f, domain, budget, rng, local_cap=80, initial_step=0.5, min_step=0.001):
    domain = _check_search_inputs(domain, budget)
    _check_search_inputs(domain, local_cap)
    history = []
    best_x, best_value = None, np.inf
    starts, used_per_start = [], []
    proposals = 0
    while len(history) < budget:
        start = rng.uniform(domain[:, 0], domain[:, 1])
        cap = min(local_cap, budget - len(history))
        run = coordinate_search(f, domain, start, cap, initial_step, min_step)
        starts.append(start.copy())
        used_per_start.append(run['evaluations'])
        proposals += run['proposals']
        if run['value'] < best_value:
            best_x = run['x'].copy()
        for local_best in run['history']:
            best_value = min(best_value, local_best)
            history.append(best_value)
    return dict(x=best_x, value=best_value, history=np.array(history),
                evaluations=len(history), starts=len(starts), proposals=proposals,
                start_points=np.array(starts), used_per_start=np.array(used_per_start))

Uniform search spends one objective call on each independent candidate.
Multistart spends some calls improving a start and then draws another start.
For random evaluated points $X_1,\ldots,X_B$, with integer budget $B\geq1$,
the retained value is $M_t=\min_{1\leq i\leq t}f(X_i)$ for $1\leq t\leq B$.
The array `history` records its observed values $b_t=M_t(\omega)$ for the
realized run $\omega$; the global best is carried across all starts.
Thus $b_{t+1}\leq b_t$ for $t<B$. Calls at repeated points
still cost one evaluation: this implementation has no cache.

**Check (S3.3):** Initial x=0.9, domain [−1,1], f(x)=(x−0.3)²,
step 0.5, local cap 5. Predict the evaluated points and the returned value.
The counter wrapper below counts actual calls independently of the solver.

In [ ]:
trace_points = []
def trace_objective(x):
    trace_points.append(float(x[0]))
    return (x[0] - 0.3)**2
trace_run = coordinate_search(trace_objective, np.array([[-1.0, 1.0]]),
                              np.array([0.9]), 5)
print('Evaluated points:', np.round(trace_points, 6))
print('Retained best values:', np.round(trace_run['history'], 6))
print('Returned point and value:', trace_run['x'], trace_run['value'])
assert len(trace_points) == trace_run['evaluations'] == 5
assert np.allclose(trace_points, [0.9, 0.4, -0.1, 0.9, 0.15])

## C. Equal evaluation budgets and repeated runs

Before running, predict on which objective local improvement will help and
why restarts may help with a multimodal function. The comparison uses:

- both declared objectives and the same box, 400 objective calls per run;
- 24 repetitions per method and objective, each with a distinct child stream;
- uniform search, multistart with local cap 40, and multistart with cap 80;
- step 0.5, minimum step 0.001, and success threshold f(x) <= 0.001.

The reference value and threshold evaluate results; the algorithms do not stop
when they reach that threshold. A shorter local run releases budget for another
start. The last local run may receive a smaller cap. This makes 400 calls an
exact total for every method. An equal call budget does not imply equal runtime.

In [ ]:
evaluation_budget = 400
repetitions = 24
root_seed = 3030
success_threshold = 0.001
problems = [('Sphere', sphere), ('Rastrigin', rastrigin)]
methods = [('Uniform', None), ('Multistart 40', 40), ('Multistart 80', 80)]
children = np.random.SeedSequence(root_seed).spawn(len(problems)*len(methods)*repetitions)
session3_results = {}
stream_index = 0
for problem_name, f in problems:
    for method_name, cap in methods:
        runs = []
        for _ in range(repetitions):
            rng = np.random.default_rng(children[stream_index])
            stream_index += 1
            if cap is None:
                run = uniform_search(f, session3_domain, evaluation_budget, rng)
            else:
                run = multistart_search(f, session3_domain, evaluation_budget, rng, local_cap=cap)
            assert run['evaluations'] == len(run['history']) == evaluation_budget
            assert np.all(np.diff(run['history']) <= 1e-12)
            runs.append(run)
        session3_results[(problem_name, method_name)] = runs
        final = np.array([run['value'] for run in runs])
        q25, median, q75 = np.quantile(final, [0.25, 0.5, 0.75])
        starts = np.array([run['starts'] for run in runs])
        print(f'{problem_name:9s} | {method_name:14s} | median={median:.6g} quartiles=[{q25:.6g},{q75:.6g}] '
              f'| successes={np.count_nonzero(final <= success_threshold)}/{repetitions} '
              f'| starts median={np.median(starts):g}')

fig_comparison, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['#246a91', '#b65e2f', '#39805e']
for ax, (problem_name, _) in zip(axes, problems):
    for (method_name, _), color in zip(methods, colors):
        histories = np.array([run['history'] for run in session3_results[(problem_name, method_name)]])
        low, median, high = np.quantile(histories, [0.25, 0.5, 0.75], axis=0)
        # A tiny display floor allows an exact zero on a logarithmic axis.
        ax.plot(np.arange(1, evaluation_budget+1), np.maximum(median, 1e-12), label=method_name, color=color)
        ax.fill_between(np.arange(1, evaluation_budget+1), np.maximum(low, 1e-12),
                        np.maximum(high, 1e-12), color=color, alpha=0.13)
    ax.set(xlabel='Objective evaluations', ylabel='Retained best value (log scale)',
           yscale='log', title=problem_name)
    ax.legend(fontsize=8)
fig_comparison.tight_layout()
plt.show()

Lines show medians; shaded bands span the first and third empirical quartiles
$Q_1,Q_3$. We use NumPy's `method='linear'` quantile convention (defined in the
glossary). The interquartile range is the width $Q_3-Q_1$; the band $[Q_1,Q_3]$
is not a confidence interval. The plot floors values at 1e−12 for display only. Numerical summaries
use the original values. All 24 runs contribute, including failures.

For the local method, subsequent points are determined by earlier evaluations,
not modeled as iid uniform draws. Do not apply $1-(1-p)^B$ to all $B$ calls.
For a fixed $k\in\mathbb N$ iid uniform starts $Z_1,\ldots,Z_k$ and the same
Borel-measurable deterministic local map $L_c:D\to D$ with cap $c\in\mathbb N$,
put $q_{\mathrm{start}}=\Pr(L_c(Z_1)\in A)$. At least one returned point is
in $A$ with probability $1-(1-q_{\mathrm{start}})^k$.
For a sublevel target $A=\{x\in D:f(x)\leq f_*+\varepsilon\}$,
$\varepsilon>0$, a run hits $A$ exactly when its retained point lies in $A$.
Early stops and a shortened last run complicate the fixed-budget model:
the number of starts can depend on previous random starts, and the last cap
can differ. The fixed-k formula cannot be applied by simply substituting the
observed number of starts.

Complete repetitions are assumed independent in the probability model.
Distinct pseudorandom child streams approximate that model; their distinct
labels do not prove probabilistic independence.

## D. Your interpretation and one bounded change

The supplied comparison already contrasts caps 40 and 80. Within the
25-minute notebook allocation:

1. State which settings are fixed and which changed.
2. Compare final medians, spread, success counts, and the number of starts for
   each objective. A count out of 24 is a rough estimate, not a guarantee.
3. Explain the exploration versus local-improvement tradeoff. Distinct streams
   mean the comparison is unpaired and includes repetition noise.
4. Inspect one run's evaluation ledger in the next cell.
5. Preserve the baseline summary, change only root_seed from 3030 to 3031
   in section C, and rerun that experiment cell. Keep all other settings fixed.
   Compare one summary and explain which conclusions remain tentative.

Do not run an additional parameter sweep. If setup is blocked, use the supplied
notes table and predict what a fresh seed could change; label predictions and
supplied evidence clearly.

In [ ]:
example_run = session3_results[('Rastrigin', 'Multistart 40')][0]
print('Calls per start:', example_run['used_per_start'])
print('Sum of calls:', example_run['used_per_start'].sum())
print('Local proposals (including infeasible ones):', example_run['proposals'])
assert example_run['used_per_start'].sum() == evaluation_budget
assert np.isclose(rastrigin(example_run['x']), example_run['value'])

**Stop here for Session 3.** In the ten-minute discussion block, compare one
claim with a peer and write your own three-sentence explanation of the success
formula's assumptions, the budget tradeoff, and one limitation of the comparison.
Use the course collaboration link announced in ELO
if you choose to share it. This reuses the assigned explanation; it adds no report.
If Python is unavailable, interpret the supplied notes table and label it as
supplied evidence.

Sphere and Rastrigin are standard test functions; these experiments use a
declared shift and domain. See the SFU
[Sphere](https://www.sfu.ca/~ssurjano/spheref.html) and
[Rastrigin](https://www.sfu.ca/~ssurjano/rastr.html) definitions and
[NumPy child streams](https://numpy.org/doc/stable/reference/random/parallel.html).
These two objectives and one budget do not establish a universally best method.

# Later sessions: random search and annealing

**Stop here for Session 3.** The remaining cells support the later annealing
sessions. O1 is covered by Session 3; O2 is covered by Session 5 using the
separate short notebook 01a. This larger comparison belongs to Session 7;
work on O3–O5 only when assigned.

These later demonstrations compare uniform search and annealing on the shifted
Rastrigin objective, using 1200 evaluations and 16 repetitions. They are a
separate experiment from today's 400-evaluation comparison.
Identify supplied implementations as reused code in any permitted project use.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

target = np.array([0.4, -0.6])
bounds = np.array([[-4.0, 4.0], [-4.0, 4.0]])

def objective(x):
    z = np.asarray(x) - target
    return float(np.sum(z*z + 10.0*(1.0 - np.cos(2.0*np.pi*z))))

print('Known minimum value:', objective(target))

## Inspect the problem
Let $D=[-4,4]^2$, $x^*=(0.4,-0.6)^T$, and
$f(x)=\sum_{j=1}^2[(x_j-x_j^*)^2+10(1-\cos(2\pi(x_j-x_j^*)))]$.
Here $f:D\to\mathbb R$, $j$ indexes coordinates, and $\pi$ is the circle constant.
Every summand is nonnegative; zero forces $x=x^*$, so $f_*=0$ is the unique
global minimum. `target` stores $x^*$ for reference; search does not use it to
choose candidates. Predict an advantage and limitation of uniform search.

The later annealer uses $y=x+sZ$, $Z\sim\mathcal N(0,I_2)$, $s=0.35$;
$I_2$ is the identity matrix and the coordinates of $Z$ are independent.
For feasible $y$, use $\Delta=f(y)-f(x)$ and acceptance
$\min\{1,e^{-\Delta/T}\}$, $T>0$. The Gaussian proposal is symmetric;
invalid proposals are rejected without calling $f$. Current state and retained
best differ. `history` stores $b_t$, the minimum over the first $t$ objective calls.
The notebook uses proposal index $k=0,1,\ldots$ and
$T_k=\max\{8(0.995)^k,10^{-12}\}$, including infeasible proposals in the index.
This numerical floor and schedule differ from OPT-01's evaluation-indexed schedule.

In [ ]:
grid = np.linspace(-4, 4, 160)
xx, yy = np.meshgrid(grid, grid)
zz = (xx-target[0])**2 + (yy-target[1])**2 + 20 - 10*np.cos(2*np.pi*(xx-target[0])) - 10*np.cos(2*np.pi*(yy-target[1]))
fig, ax = plt.subplots(figsize=(5.5, 4))
contour = ax.contourf(xx, yy, zz, levels=25, cmap='viridis')
ax.plot(*target, 'rx', markersize=8, label='Known optimum')
ax.set(xlabel='x1', ylabel='x2', title='A continuous multimodal objective')
ax.legend()
fig.colorbar(contour, ax=ax, label='Objective value')
fig.tight_layout()
plt.show()

In [ ]:
def random_search(f, domain, budget, rng):
    best_x = None
    best_value = np.inf
    history = []
    for _ in range(budget):
        x = rng.uniform(domain[:, 0], domain[:, 1])
        value = f(x)
        if value < best_value:
            best_x, best_value = x.copy(), value
        history.append(best_value)
    return {'x': best_x, 'value': best_value, 'history': np.asarray(history), 'evaluations': budget}

def annealing(f, domain, budget, rng, initial_temperature=8.0, cooling=0.995, proposal_scale=0.35):
    if budget < 1 or initial_temperature <= 0 or not 0 < cooling <= 1 or proposal_scale <= 0:
        raise ValueError('Use a positive budget, temperature and scale, with 0 < cooling <= 1.')
    current = rng.uniform(domain[:, 0], domain[:, 1])
    current_value = f(current)
    best_x, best_value = current.copy(), current_value
    history = [best_value]
    evaluations = 1
    # Invalid proposals are rejected; they do not consume an objective evaluation.
    # The temperature schedule advances for every proposal, including a rejection.
    for proposal_index in range(100 * budget):
        if evaluations == budget:
            break
        temperature = max(initial_temperature * cooling**proposal_index, 1e-12)
        proposal = current + rng.normal(scale=proposal_scale, size=current.size)
        if np.any(proposal < domain[:, 0]) or np.any(proposal > domain[:, 1]):
            continue
        proposal_value = f(proposal)
        evaluations += 1
        change = proposal_value - current_value
        if change <= 0 or rng.random() < np.exp(-change / temperature):
            current, current_value = proposal, proposal_value
        if current_value < best_value:
            best_x, best_value = current.copy(), current_value
        history.append(best_value)
    if evaluations != budget:
        raise RuntimeError('Too many invalid proposals; inspect the proposal scale and domain.')
    return {'x': best_x, 'value': best_value, 'history': np.asarray(history), 'evaluations': evaluations}

## Compare typical behaviour
Both methods receive $B=1200$ objective calls, including initialization, and
the same domain. Feasible rejected candidates count; invalid proposals do not.
The guard raises an error if the budget cannot be completed within $100B$
proposals. The model assumes $R=16$ independent whole runs per method;
distinct child streams approximate this model and do not prove independence.
Predict the curves. Their shaded quartile bands span $[Q_1,Q_3]$ under the
glossary's linear empirical-quantile convention. IQR is the width $Q_3-Q_1$;
the band is not a confidence interval for the median.

In [ ]:
budget = 1200
repetitions = 16
seeds = np.random.SeedSequence(31415).spawn(2 * repetitions)
random_runs, annealing_runs = [], []
for k in range(repetitions):
    random_runs.append(random_search(objective, bounds, budget, np.random.default_rng(seeds[2*k]))['history'])
    annealing_runs.append(annealing(objective, bounds, budget, np.random.default_rng(seeds[2*k+1]))['history'])

fig, ax = plt.subplots(figsize=(7, 4))
x_axis = np.arange(1, budget + 1)
for label, runs, color in [('Random search', random_runs, '#36789a'), ('Annealing', annealing_runs, '#c56a32')]:
    values = np.asarray(runs)
    low, median, high = np.quantile(values, [0.25, 0.5, 0.75], axis=0)
    ax.plot(x_axis, median, label=label, color=color)
    ax.fill_between(x_axis, low, high, color=color, alpha=0.18)
    print(f'{label}: median final value={median[-1]:.4f}; quartiles=[{low[-1]:.4f}, {high[-1]:.4f}]')
ax.set(xlabel='Objective evaluations', ylabel='Best objective value found', title='Equal budgets and repeated runs')
ax.legend()
fig.tight_layout()
plt.show()

## Your investigation
Recall O1 from Session 3 and O2 from Session 5; complete O3 when this later route is assigned. Count overlapping work once. For O4, vary either cooling or proposal scale while keeping the other settings fixed. Repeat the experiments and explain a failure case. Use O5 to review a peer's evidence. Optional: replace the objective with a smooth quadratic and explain why the conclusion can change.

The simple cooling rule is a practical heuristic. This notebook does not demonstrate a general global-convergence theorem. It will be revisited when we compare fixed-temperature sampling with annealing.